# 🔍 Notebook 4 — Keyword Spotting + Summarization
**Task A:** Detect specific Arabic keywords in transcribed speech
**Task B:** Summarize Arabic transcripts using mT5
**Hardware:** Kaggle T4 GPU


## Step 1 — Clean Disk

In [1]:
import os, shutil

print('🧹 Cleaning up disk space...')
dirs_to_clean = ['/kaggle/working/hf_cache']
for path in dirs_to_clean:
    try:
        if os.path.isdir(path): shutil.rmtree(path)
    except: pass

total, used, free = shutil.disk_usage('/kaggle/working')
print(f'💾 Free: {free/1e9:.1f} GB')
print('✅ Ready')

🧹 Cleaning up disk space...
💾 Free: 20.9 GB
✅ Ready


## Step 2 — Install Dependencies

In [2]:
!pip install -q transformers datasets sentence-transformers
!pip install -q torch librosa soundfile
print('✅ All packages installed')

✅ All packages installed


## Step 3 — Kaggle Cache Fix

In [4]:
import os

os.makedirs('/kaggle/working/hf_cache', exist_ok=True)
os.makedirs('/kaggle/working/hf_cache/datasets', exist_ok=True)
os.makedirs('/kaggle/working/hf_cache/hub', exist_ok=True)

os.environ['HF_HOME']               = '/kaggle/working/hf_cache'
os.environ['HF_DATASETS_CACHE']     = '/kaggle/working/hf_cache/datasets'
os.environ['TRANSFORMERS_CACHE']    = '/kaggle/working/hf_cache/hub'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/kaggle/working/hf_cache/hub'

print('✅ HuggingFace cache redirected to /kaggle/working/hf_cache')

✅ HuggingFace cache redirected to /kaggle/working/hf_cache


## Step 4 — GPU Check

In [5]:
import torch
import numpy as np
import re
from typing import List, Dict

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla P100-PCIE-16GB


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


## Step 5 — Keyword Spotting (Rule-Based)

In [6]:
KEYWORD_GROUPS = {
    'emergency': {
        'en': 'Emergency', 'color': '🔴',
        'keywords': ['طوارئ', 'نجدة', 'استغاثة', 'خطر', 'حادث', 'أزمة']
    },
    'deadline': {
        'en': 'Deadline', 'color': '🟡',
        'keywords': ['موعد', 'مهلة', 'الموعد النهائي', 'تسليم', 'التسليم']
    },
    'exam': {
        'en': 'Exam', 'color': '🟠',
        'keywords': ['امتحان', 'اختبار', 'مذاكرة', 'درجة', 'نجاح', 'رسوب']
    },
    'meeting': {
        'en': 'Meeting', 'color': '🔵',
        'keywords': ['اجتماع', 'مقابلة', 'تجمع', 'ندوة', 'مؤتمر']
    },
    'important': {
        'en': 'Important', 'color': '🟣',
        'keywords': ['مهم', 'ضروري', 'عاجل', 'أساسي', 'حاسم']
    }
}

def normalize_arabic_text(text: str) -> str:
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F]', '', text)
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'[ىئ]', 'ي', text)
    text = text.replace('ؤ', 'و').replace('ة', 'ه')
    return text

def spot_keywords(transcript: str) -> List[Dict]:
    normalized = normalize_arabic_text(transcript)
    matches = []
    for category, info in KEYWORD_GROUPS.items():
        for kw in info['keywords']:
            norm_kw = normalize_arabic_text(kw)
            if norm_kw in normalized:
                matches.append({
                    'keyword':     kw,
                    'category':    category,
                    'category_en': info['en'],
                    'color':       info['color'],
                })
    return matches

# Test
test = 'يجب أن تعرف أن هناك امتحانًا مهمًا غدًا وموعد التسليم بعد أسبوع'
matches = spot_keywords(test)
print(f'Test: {test}')
print(f'Found {len(matches)} keyword(s):')
for m in matches:
    print(f'  {m["color"]} [{m["category_en"]}] "{m["keyword"]}"')
print('✅ Keyword spotting ready')

Test: يجب أن تعرف أن هناك امتحانًا مهمًا غدًا وموعد التسليم بعد أسبوع
Found 5 keyword(s):
  🟡 [Deadline] "موعد"
  🟡 [Deadline] "تسليم"
  🟡 [Deadline] "التسليم"
  🟠 [Exam] "امتحان"
  🟣 [Important] "مهم"
✅ Keyword spotting ready


## Step 6 — Semantic Keyword Search (Multilingual Embeddings)

In [8]:
from sentence_transformers import SentenceTransformer

print('Loading multilingual sentence encoder...')
# Force CPU to avoid CUDA kernel compatibility issue with T4
encoder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device='cpu')
print('✅ Encoder loaded on CPU')

SEMANTIC_ANCHORS = {
    'emergency': ['طوارئ', 'نجدة', 'استغاثة', 'خطر', 'حادث'],
    'deadline':  ['موعد', 'مهلة', 'الموعد النهائي', 'تسليم'],
    'exam':      ['امتحان', 'اختبار', 'مذاكرة', 'درجة', 'نجاح'],
    'meeting':   ['اجتماع', 'مقابلة', 'تجمع', 'ندوة', 'مؤتمر'],
    'important': ['مهم', 'ضروري', 'عاجل', 'أساسي', 'حاسم'],
}

anchor_embeddings = {}
for category, phrases in SEMANTIC_ANCHORS.items():
    embs = encoder.encode(phrases, normalize_embeddings=True)
    anchor_embeddings[category] = embs.mean(axis=0)

print(f'✅ Anchor embeddings ready: {list(anchor_embeddings.keys())}')

Loading multilingual sentence encoder...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Encoder loaded on CPU
✅ Anchor embeddings ready: ['emergency', 'deadline', 'exam', 'meeting', 'important']


## Step 7 — Apply Keyword Spotting to FLEURS Transcripts

In [10]:
import numpy as np

def semantic_keyword_score(text: str, threshold: float = 0.4) -> dict:
    """Return categories whose semantic similarity exceeds threshold."""
    if not text or not text.strip():
        return {}
    try:
        text_emb = encoder.encode([text], normalize_embeddings=True)[0]
        matches  = {}
        for category, anchor_emb in anchor_embeddings.items():
            score = float(np.dot(text_emb, anchor_emb))
            if score >= threshold:
                matches[category] = round(score, 3)
        return matches
    except Exception:
        return {}

print('✅ semantic_keyword_score ready')

✅ semantic_keyword_score ready


In [11]:
from datasets import load_dataset

print('Loading FLEURS ar_eg test split...')
fleurs_test = load_dataset('google/fleurs', 'ar_eg', split='test')

print(f'\n🔍 Keyword Spotting on {len(fleurs_test)} FLEURS transcripts:')
print('=' * 60)
found_any = False

for i, sample in enumerate(fleurs_test):
    transcript       = sample['transcription']
    exact_matches    = spot_keywords(transcript)
    semantic_matches = semantic_keyword_score(transcript)

    if exact_matches or semantic_matches:
        found_any = True
        print(f'\nSample {i}: {transcript}')
        for m in exact_matches:
            print(f'  {m["color"]} Exact:    [{m["category_en"]}] "{m["keyword"]}"')
        for cat, score in semantic_matches.items():
            print(f'  🔷 Semantic: [{cat}] score={score}')

if not found_any:
    print('No target keywords found.')
    print('FLEURS is read-aloud news text — not spontaneous speech.')
    print('The system works correctly — test it with your own audio.')

print('\n✅ Keyword spotting on FLEURS done')

Loading FLEURS ar_eg test split...

🔍 Keyword Spotting on 428 FLEURS transcripts:

Sample 20: السبائك هي بشكل أساسي خليط من اثنين أو أكثر من المعادن لا تنسَ أن هناك العديد من العناصر في الجدول الدوري
  🟣 Exact:    [Important] "أساسي"

Sample 26: لم يصدر تحذير عن تسونامي ووفقا لوكالة جاكرتا للجيوفيزياء لن يصدر تحذير عن تسونامي لأن الزلزال لم يستوف متطلبات 6.5 درجة
  🟠 Exact:    [Exam] "درجة"

Sample 30: الأسود هي أكثر القطط اجتماعية وتعيش في مجموعات كبيرة تدعى قطعاناً
  🔵 Exact:    [Meeting] "اجتماع"

Sample 55: تم عمل الإعلان بعد أن قام ترامب بمحادثة هاتفية مع الرئيس التركي رجب الطيب أردوغان
  🔴 Exact:    [Emergency] "حادث"

Sample 99: ومن بين أكثر الطرق شيوعاً التي تستخدم لتوضيح أهمية التنشئة الاجتماعية الاعتماد على الحالات القليلة المؤسفة للأطفال الذين عانوا من خلال الإهمال أو سوء الحظ أو الإيذاء المتعمد غير مرتبطين اجتماعياً من جانب البالغين أثناء نشأتهم
  🔵 Exact:    [Meeting] "اجتماع"

Sample 100: تشمل الموضوعات الأخرى المدرجة على جدول أعمال بالي الحفاظ على الغابات المتبقية في الع

## Step 8 — Arabic Summarization with mT5

In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SUMM_MODEL = 'csebuetnlp/mT5_multilingual_XLSum'
print(f'Loading {SUMM_MODEL}...')
tokenizer_summ = AutoTokenizer.from_pretrained(SUMM_MODEL)

# Force CPU — mT5 has CUDA kernel incompatibility with T4 on this session
model_summ = AutoModelForSeq2SeqLM.from_pretrained(SUMM_MODEL).to('cpu')
model_summ.eval()
print('✅ Summarization model loaded on CPU')

def summarize_arabic(text: str, max_new_tokens: int = 80) -> str:
    if len(text.strip()) < 50:
        return '(Text too short to summarize)'
    inputs = tokenizer_summ(
        text.strip(), return_tensors='pt',
        padding='max_length', truncation=True, max_length=512
    ).to('cpu')
    with torch.no_grad():
        output_ids = model_summ.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            num_beams=4, max_new_tokens=max_new_tokens,
            early_stopping=True, no_repeat_ngram_size=3,
        )
    return tokenizer_summ.decode(output_ids[0], skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)

# Test
test_text = """
أعلنت وزارة التعليم عن جدول امتحانات الفصل الدراسي الثاني لطلاب المرحلة الثانوية.
ستبدأ الامتحانات في الخامس عشر من الشهر القادم وستستمر حتى نهاية الشهر.
يجب على جميع الطلاب الحضور في الموعد المحدد وإحضار بطاقة الطالب.
في حالة الغياب دون عذر مقبول، لن يتمكن الطالب من إعادة الامتحان.
كما أكدت الوزارة على أهمية المذاكرة الجيدة والاستعداد المبكر لضمان النجاح.
"""
print('Input text:')
print(test_text)
print('=' * 60)
summary = summarize_arabic(test_text)
print('Summary:')
print(summary)
print(f'\nCompression: {len(test_text)} chars → {len(summary)} chars ({len(summary)/len(test_text)*100:.1f}%)')

Loading csebuetnlp/mT5_multilingual_XLSum...


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Summarization model loaded on CPU
Input text:

أعلنت وزارة التعليم عن جدول امتحانات الفصل الدراسي الثاني لطلاب المرحلة الثانوية.
ستبدأ الامتحانات في الخامس عشر من الشهر القادم وستستمر حتى نهاية الشهر.
يجب على جميع الطلاب الحضور في الموعد المحدد وإحضار بطاقة الطالب.
في حالة الغياب دون عذر مقبول، لن يتمكن الطالب من إعادة الامتحان.
كما أكدت الوزارة على أهمية المذاكرة الجيدة والاستعداد المبكر لضمان النجاح.

Summary:
أعلنت وزارة التعليم في مصر عن جدول امتحانات الفصل الدراسي الثاني لطلاب المرحلة الثانوية.

Compression: 360 chars → 88 chars (24.4%)


## Step 9 — Summarize FLEURS Transcripts (Batch Demo)

In [14]:
# Combine 10 FLEURS transcripts and summarize
combined_text = ' '.join([
    fleurs_test[i]['transcription'] for i in range(min(10, len(fleurs_test)))
])

print(f'Combined transcript ({len(combined_text)} chars):')
print(combined_text[:400] + '...')
print()

summary = summarize_arabic(combined_text)
print('Summary:')
print(summary)
print(f'\nCompression: {len(combined_text)} → {len(summary)} chars ({len(summary)/len(combined_text)*100:.1f}%)')

Combined transcript (1280 chars):
تشكلت في المحيط الأطلسي اليوم عاشر عاصفة مُسماة لموسم الأعاصير الأطلسية العاصفة شبه الاستوائية جيري تغادر الحافلات المحطة الداخلية بين المناطق عبر النهر في خلال اليوم على الرغم من أن معظمها وخاصة المتجهة منها إلى الشرق وجاكار/بومثانج تغادر بين 06:30 و 07:30 يتمُّ دعم التعلُّم التفاعليّ في البرنامج داخليًا ويهدف إلى طرح الأسئلة والتحفيز وشرح الإجراءاتِ التي قد يكون من الصعب على الطالب التعامل معها ...

Summary:
تشهد المحيط الأطلسي اليوم عاصفة مُسماة لموسم الأعاصير الأطلسية .

Compression: 1280 → 64 chars (5.0%)
